# Attack run

Runs 60 prompt injection attacks against four system configurations, three runs each.
Same setup as evaluation_run: vLLM on the GPU, dbt + DuckDB + MCP server alongside.
Needs an A100 (80 GB) with high RAM.

Configurations from the pre-registration:
1. `multi` — full pipeline, all defense layers
2. `multi-rules` — reviewer ablated, auto-approve on rules + dry_run pass
3. `multi --defense spotlight` — tool output wrapped in `<data>` tags
4. `single` — single-agent baseline

Dev split (40 attacks) for development; test split (20 attacks) run once after analysis.

In [ ]:
import os, subprocess

repo = "/content/responder"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/arihaaant/data-incident-responder.git", repo], check=True)
print("code at", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/responder
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

In [ ]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

## Model server

In [ ]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

## Warehouse and tests

In [ ]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "test suite failed"

## Validate attacks

Quick check on a few attacks to confirm placement doesn't change the base incident answers.

In [ ]:
!python -m responder.kit --validate-attacks --only atk001,atk010,atk025,atk040,atk050,atk060

## Drive mount

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "attack-final"
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

## Attack runs — dev split, three runs per config

Four configs × 3 runs × 40 dev attacks = 480 incident runs.
Each incident that already has a result file is skipped, so this cell is safe to re-run.

In [ ]:
configs = [
    ("multi",       None),
    ("multi-rules", None),
    ("multi",       "spotlight"),
    ("single",      None),
]

for run in range(1, 4):
    for system, defense in configs:
        tag = f"{system}+{defense}" if defense else system
        folder = f"{out}/{tag}-{run}"
        defense_flag = f"--defense {defense}" if defense else ""
        print(f"\n=== {tag}, run {run} ===")
        !python -m responder.run --set attacks --split dev --system {system} {defense_flag} --workers 6 --out {folder}

## Test split — one run per config

Run only after dev analysis is done. 20 test attacks × 4 configs = 80 runs.

In [ ]:
for system, defense in configs:
    tag = f"{system}+{defense}" if defense else system
    folder = f"{out}/{tag}-test"
    defense_flag = f"--defense {defense}" if defense else ""
    print(f"\n=== {tag}, test ===")
    !python -m responder.run --set attacks --split test --system {system} {defense_flag} --workers 6 --out {folder}

## Results

In [ ]:
!python -m responder.attack_analysis {out}

In [ ]:
import json
from pathlib import Path

result = json.loads((Path(out) / "attack_compare.json").read_text())

print(f"{'system':20s} {'echoed':>8s} {'attempted':>10s} {'reached':>8s} {'resolved':>9s}")
print("-" * 60)
for system in ["multi", "multi-rules", "multi+spotlight", "single"]:
    r = result["rates"].get(system, {})
    res = result["resolution"].get(system, {})
    if r:
        print(f"{system:20s} {100*r['echoed']:7.1f}% {100*r['attempted']:9.1f}% {100*r['reached']:7.1f}% {100*res.get('rate',0):8.1f}%")

print("\nstopped-by distribution:")
for system, dist in result.get("stopped_by", {}).items():
    layers = ", ".join(f"{l}: {100*v['rate']:.0f}%" for l, v in dist.items())
    print(f"  {system}: {layers}")

print("\npaired comparisons:")
for name, comp in result.get("comparisons", {}).items():
    print(f"  {name}: {comp['a_only']} vs {comp['b_only']}, p = {comp['p']:.3f}")